In [1]:
from pathlib import Path
import os
import sys

project_root = Path.cwd()
if not (project_root / 'database').exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))
os.environ['DATABASE_URL'] = f"sqlite:///{project_root / 'similar_songs.db'}"

print(f'Project root: {project_root}')
print(f'Database found: {(project_root / "similar_songs.db").exists()}')

Project root: c:\programming\Projects\Music_Mind
Database found: True


# Phase 4 - Similar Songs Recommendation System

This phase builds a content-based recommender from the Phase 3 SQLite database. It standardizes the audio features, computes cosine similarity, and returns tracks with similar musical profiles.

Popularity is kept for display, but it is not used as a similarity feature so recommendations are driven by sound rather than catalog popularity.

## 1. Load the recommender

The recommender loads one row per unique track from the database and eagerly loads genre and artist relationships.

In [2]:
from dataclasses import dataclass

import numpy as np
from sklearn.preprocessing import StandardScaler, normalize
from sqlalchemy.orm import joinedload

from database.db.database import SessionLocal
from database.db.models import Track

FEATURE_COLUMNS = (
    'danceability', 'energy', 'loudness', 'speechiness', 'acousticness',
    'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms'
)

@dataclass(frozen=True)
class Recommendation:
    track_id: str
    track_name: str
    artists: tuple
    genres: tuple
    popularity: int | None
    similarity: float

class SimilarityRecommender:
    def __init__(self):
        with SessionLocal() as session:
            self.tracks = (session.query(Track)
                           .options(joinedload(Track.artists), joinedload(Track.genres))
                           .all())
        if not self.tracks:
            raise RuntimeError('The database contains no tracks.')
        raw_features = np.array(
            [[getattr(track, column) for column in FEATURE_COLUMNS] for track in self.tracks],
            dtype=float,
        )
        medians = np.nanmedian(raw_features, axis=0)
        raw_features = np.where(np.isfinite(raw_features), raw_features, medians)
        self.matrix = normalize(StandardScaler().fit_transform(raw_features))
        self.by_id = {track.track_id: index for index, track in enumerate(self.tracks)}

    def search_tracks(self, query, limit=10):
        query = query.strip().casefold()
        matches = []
        for track in self.tracks:
            artists = ' '.join(item.name for item in track.artists)
            if query and (query in track.track_name.casefold() or query in artists.casefold()):
                matches.append(track)
                if len(matches) == limit:
                    break
        return matches

    def recommend(self, track_id, limit=10, genre=None, artist=None):
        if track_id not in self.by_id:
            raise KeyError(f'Unknown track_id: {track_id}')
        source_index = self.by_id[track_id]
        scores = self.matrix @ self.matrix[source_index]
        candidates = [index for index in range(len(self.tracks)) if index != source_index]
        if genre:
            candidates = [index for index in candidates if any(item.name.casefold() == genre.casefold() for item in self.tracks[index].genres)]
        if artist:
            candidates = [index for index in candidates if any(item.name.casefold() == artist.casefold() for item in self.tracks[index].artists)]
        ranked = sorted(candidates, key=lambda index: scores[index], reverse=True)[:limit]
        return [Recommendation(
            track_id=self.tracks[index].track_id,
            track_name=self.tracks[index].track_name,
            artists=tuple(item.name for item in self.tracks[index].artists),
            genres=tuple(item.name for item in self.tracks[index].genres),
            popularity=self.tracks[index].popularity,
            similarity=float(scores[index]),
        ) for index in ranked]

recommender = SimilarityRecommender()
print(f'Loaded tracks: {len(recommender.tracks):,}')
print(f'Features: {list(FEATURE_COLUMNS)}')

Loaded tracks: 89,740
Features: ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms']


## 2. Feature preparation

The audio features have different units and ranges. StandardScaler puts them on comparable scales, and row normalization makes the dot product equivalent to cosine similarity. Missing or infinite values are replaced with feature medians.

In [3]:
from sklearn.preprocessing import normalize

print(f'Similarity matrix shape: {recommender.matrix.shape}')
print(f'Every vector is normalized: {normalize(recommender.matrix[:3]).shape == recommender.matrix[:3].shape}')

Similarity matrix shape: (89740, 10)
Every vector is normalized: True


## 3. Find a seed track

Search by part of a track title or artist name, then choose a result as the seed for recommendations.

In [4]:
seed_matches = recommender.search_tracks('Comedy', limit=5)
for index, match in enumerate(seed_matches, start=1):
    artists = ', '.join(item.name for item in match.artists)
    print(f"{index}. {match.track_name} - {artists} [{match.track_id}]")

1. Comedy - Gen Hoshino [5SuOikwiRyPMVoIQDJUgSV]
2. The Best Comedy I've Ever Seen - Patton Oswalt [0oISYk7B6GJK0euROwbfKn]
3. Accidental Soccer Coach - Dry Bar Comedy [5g9IN42ZX2NK6KhOB347V3]
4. Old People's Music - Dry Bar Comedy [4F6BynYVLvYxM6q1gMDdvz]


## 4. Generate similar songs

The source track is excluded automatically. Similarity scores closer to 1 indicate closer audio-feature profiles.

In [5]:
seed_id = seed_matches[0].track_id
recommendations = recommender.recommend(seed_id, limit=10)

for index, recommendation in enumerate(recommendations, start=1):
    artists = ', '.join(recommendation.artists)
    print(f"{index:2}. {recommendation.similarity:.3f}  {recommendation.track_name} - {artists}")

 1. 0.954  Skylight - Gramatik
 2. 0.932  Monde de fous - Danakil
 3. 0.931  Look For The Good (Single Version) - Jason Mraz
 4. 0.919  Mi Eden de Tristeza - Porta, Pumpkin
 5. 0.913  Pop Virus - Gen Hoshino
 6. 0.912  The Gambler - Busy Signal
 7. 0.904  Bu Aşk Olur Mu - MFÖ
 8. 0.902  That Certain Female - Charlie Feathers
 9. 0.897  JAMAICA - Feid, Sech
10. 0.892  Hold Me Tight - The Green


## 5. Filter recommendations

Genre and artist filters are applied after the source track is removed. This allows queries such as similar songs within a particular genre.

In [6]:
kpop_recommendations = recommender.recommend(seed_id, limit=5, genre='k-pop')

for recommendation in kpop_recommendations:
    print(f"{recommendation.similarity:.3f}  {recommendation.track_name} - {', '.join(recommendation.artists)}")

0.786  Pink Venom - BLACKPINK
0.786  Pink Venom - BLACKPINK
0.784  Trademark (From "James - Kannada") - Aditi Sagar, Chandan Shetty, Charan Raj, MC Vickey, Sharmila, Yuva Rajkumar
0.783  Naatu Sarakku - Dhanush, Ranjith Govind
0.782  Rush Hour (Feat. j-hope of BTS) - Crush, j-hope


## 6. Recommendation helper

Use this helper to search for a song and print its recommendations without manually handling track IDs.

In [7]:
def recommend_by_search(query, limit=10, genre=None):
    matches = recommender.search_tracks(query, limit=1)
    if not matches:
        raise ValueError(f'No track found for: {query}')
    seed = matches[0]
    artists = ', '.join(item.name for item in seed.artists)
    print(f"Seed: {seed.track_name} - {artists}")
    return recommender.recommend(seed.track_id, limit=limit, genre=genre)

recommend_by_search('Bad Bunny', limit=5)

Seed: I Like It - Bad Bunny, Cardi B, J Balvin


[Recommendation(track_id='58q2HKrzhC3ozto2nDdN4z', track_name='I Like It', artists=('Bad Bunny', 'Cardi B', 'J Balvin'), genres=('dance',), popularity=79, similarity=0.9999882371326256),
 Recommendation(track_id='06w2selQidraumTXgSs5sS', track_name='Chavanprash', artists=('Arohi Mhatre', 'Divya Kumar', 'Pragati Joshi'), genres=('folk',), popularity=34, similarity=0.9771006800255574),
 Recommendation(track_id='7MRDg3WyBuOO2dA7mn1IYV', track_name='Momotaro', artists=('WEDNESDAY CAMPANELLA',), genres=('j-dance',), popularity=41, similarity=0.9601923382542907),
 Recommendation(track_id='0qfsgqyonHi2uusP2jNPwT', track_name='Bolo Um', artists=('MC Neguinho do Kaxeta',), genres=('funk',), popularity=41, similarity=0.9530808724114582),
 Recommendation(track_id='5kwuPm3p11TnirbMvcGrZZ', track_name='Spooky, Scary Skeletons - Undead Tombstone Remix Extended', artists=('Andrew Gold',), genres=('power-pop',), popularity=55, similarity=0.9509071080201673)]

## Phase 4 decisions

- One vector per unique `track_id`, matching the Phase 3 deduplication strategy.
- Standardized audio features prevent tempo, duration, or loudness from dominating distance.
- Popularity is excluded from similarity and remains available for ranking or display later.
- Genre is an optional filter instead of a mandatory numeric feature.
- The next extension can expose `recommend()` through the backend API or a user interface.